# Mode-Conditioned Drawdown-Drift Reward Recipe Screen

This notebook screens a drawdown-drift plus trend-preservation reward patch for `mode_conditioned_rebound_reward_train_v2`, the current top strict retraining queue candidate.

It is training-readiness evidence only. It does not promote or retain an actor.

## Governance

- Canonical baseline: `soxl_backtest_policy_reimplementation_v1` as the qlib-aligned SOXLBacktest buy/sell reference baseline.
- Direct comparison: `mode_conditioned_rebound_reward_train_v2`.
- Role: `diagnose`.
- Target window: required official walk-forward windows `wf_2020`, `wf_2022`, and `wf_2023_2024`, plus the strict YTD return alignment gate.
- Accepted trade-off: short fold-local BC/PPO training screen only; no actor retention or promotion until official WF gate passes.
- Changed layer: `reward`.
- External reference: repo-owned drawdown-drift reward-screen helper and SOXLBacktest qlib reference baseline.
- Hypothesis: a reward patch should penalize active exposure during controllable drawdown drift while preserving participation when SOXL buy-and-hold is trending and the candidate underparticipates.
- Acceptance criteria: recipe-ready screen, controllable drawdown capture >= 0.50, trend underparticipation capture >= 0.50, overreach share <= 0.25, promotion remains false.
- Failure mode: recipe only captures one side, overreaches into drawdown, or treats flat/no-position drawdown tails as active exposure failures.

## Historical Non-Duplicate Check

- Closest success case: `short_volshock_drawdown_drift_reward_recipe_screen_v1`, which produced a recipe for a different short-volshock candidate.
- Closest failure case: `state_conditioned_overlay_gap_to_20_v1`, where simple post-hoc position throttles failed to satisfy both drawdown and return gates.
- Why this is not a duplicate: this screen is candidate-specific to `mode_conditioned_rebound_reward_train_v2`, uses the official daily paths for all required WF windows, and evaluates reward pressure rather than post-hoc caps or heuristic overlays.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "soxl_quant").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.drawdown_drift_reward_screen import (
    render_drawdown_drift_reward_screen_review,
    screen_drawdown_drift_reward_recipes,
    summarize_drawdown_drift_reward_screen,
)

EXPERIMENT = "mode_conditioned_drawdown_drift_reward_recipe_screen_v1"
PROFILE = "mode_conditioned_rebound_reward_train_v2"
CARD_DIR = ROOT / "runtime" / "reward" / "card" / EXPERIMENT
MODEL_CARD = ROOT / "runtime" / "model" / "card" / PROFILE
CARD_DIR.mkdir(parents=True, exist_ok=True)

STRICT_QUEUE = ROOT / "runtime" / "model_selection" / "card" / "strict_retraining_candidate_queue_v1" / "summary.json"
STRICT_GATE = ROOT / "runtime" / "model_selection" / "card" / "model_retention_walkforward_gate_v1" / "summary.json"
FINAL_CONTRACT = ROOT / "runtime" / "model_selection" / "card" / "final_walkforward_retention_contract_v1" / "summary.json"

required_windows = ["wf_2020", "wf_2022", "wf_2023_2024"]
daily_paths = {
    f"{PROFILE}__{window}": pd.read_csv(MODEL_CARD / f"{PROFILE}_{window}_soxl_daily.csv")
    for window in required_windows
}

strict_queue = json.loads(STRICT_QUEUE.read_text())
strict_gate = json.loads(STRICT_GATE.read_text())["summary"]
final_contract = json.loads(FINAL_CONTRACT.read_text())["summary"]

print("top_candidate", strict_queue["top_candidate_id"])
print("final_retained_count", final_contract["final_retained_count"])
print("strict_passing_profile_count", strict_gate["passing_profile_count"])


top_candidate mode_conditioned_rebound_reward_train_v2
final_retained_count 0
strict_passing_profile_count 0


In [2]:
def window_metrics(label: str, frame: pd.DataFrame) -> dict[str, object]:
    window = label.rsplit("__", 1)[1]
    strategy_nav = pd.to_numeric(frame["strategy_nav"], errors="coerce")
    buyhold_nav = pd.to_numeric(frame["buyhold_nav"], errors="coerce")
    position = pd.to_numeric(frame["position"], errors="coerce").fillna(0.0)
    strategy_return_pct = (strategy_nav.iloc[-1] / strategy_nav.iloc[0] - 1.0) * 100.0
    buyhold_return_pct = (buyhold_nav.iloc[-1] / buyhold_nav.iloc[0] - 1.0) * 100.0
    relative_return_pct = strategy_return_pct - buyhold_return_pct
    max_dd_pct = (1.0 - strategy_nav / strategy_nav.cummax()).max() * 100.0
    if max_dd_pct > 20.0 and relative_return_pct < 0.0:
        failure_type = "joint_tradeoff"
    elif max_dd_pct > 20.0:
        failure_type = "drawdown_control"
    elif relative_return_pct < 0.0:
        failure_type = "trend_capture"
    else:
        failure_type = "pass"
    return {
        "profile": PROFILE,
        "window": window,
        "rows": int(len(frame)),
        "return_pct": float(strategy_return_pct),
        "buyhold_return_pct": float(buyhold_return_pct),
        "relative_return_pct": float(relative_return_pct),
        "max_drawdown_pct": float(max_dd_pct),
        "avg_position": float(position.mean()),
        "relative_gap": float(max(0.0, 30.0 - relative_return_pct)),
        "max_dd_gap": float(max(0.0, max_dd_pct - 20.0)),
        "gate_pass": failure_type == "pass",
        "failure_type": failure_type,
    }

gate_gaps = pd.DataFrame([window_metrics(label, frame) for label, frame in daily_paths.items()])
gate_gaps.to_csv(CARD_DIR / "gate_gaps.csv", index=False)
gate_gaps

,profile,window,rows,return_pct,buyhold_return_pct,relative_return_pct,max_drawdown_pct,avg_position,relative_gap,max_dd_gap,gate_pass,failure_type
0,mode_conditioned_rebound_reward_train_v2,wf_2020,252,50.083327,58.093571,-8.010244,23.086304,0.260252,38.010244,3.086304,False,joint_tradeoff
1,mode_conditioned_rebound_reward_train_v2,wf_2022,250,-12.705282,-86.588072,73.882789,15.900697,0.091070,0.000000,0.000000,True,pass
2,mode_conditioned_rebound_reward_train_v2,wf_2023_2024,251,-16.085903,-0.356635,-15.729268,30.489188,0.273452,45.729268,10.489188,False,joint_tradeoff


In [3]:
scores = screen_drawdown_drift_reward_recipes(
    daily_paths,
    gate_gaps=gate_gaps.rename(columns={"relative_return_pct": "relative_return", "max_drawdown_pct": "max_dd"}),
    selected_candidate=PROFILE,
)
summary = summarize_drawdown_drift_reward_screen(scores)
review = render_drawdown_drift_reward_screen_review(scores, summary, top_n=12)

scores.to_csv(CARD_DIR / "reward_recipe_scores.csv", index=False)
(CARD_DIR / "reward_recipe_review.md").write_text(review, encoding="utf-8")
(CARD_DIR / "reward_objective_patch.json").write_text(
    json.dumps(summary["reward_objective_patch"], indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

selection = {
    "experiment": EXPERIMENT,
    "candidate_profile": PROFILE,
    "decision": summary["decision"],
    "selected_recipe": summary["selected_recipe"],
    "training_allowed_next": summary["training_allowed_next"],
    "promotion_allowed": summary["promotion_allowed"],
    "actor_retention_allowed": False,
    "next_mechanism": summary["next_mechanism"],
    "strict_queue_top_candidate": strict_queue["top_candidate_id"],
    "strict_final_retained_count": final_contract["final_retained_count"],
    "strict_passing_profile_count": strict_gate["passing_profile_count"],
    "risk_notes": [
        "This is a reward-objective readiness screen, not a trained actor or promotion artifact.",
        "Raw drawdown capture remains visible because no-position drawdown tails can keep max drawdown elevated after exposure is removed.",
        "Next step must run a short fold-local BC/PPO training screen and then the official WF gate.",
    ],
    "sidecars": {
        "gate_gaps": str(CARD_DIR / "gate_gaps.csv"),
        "reward_recipe_scores": str(CARD_DIR / "reward_recipe_scores.csv"),
        "reward_objective_patch": str(CARD_DIR / "reward_objective_patch.json"),
        "reward_recipe_review": str(CARD_DIR / "reward_recipe_review.md"),
        "selection": str(CARD_DIR / "selection.json"),
        "summary": str(CARD_DIR / "summary.json"),
    },
}
summary_payload = {
    "experiment": EXPERIMENT,
    "candidate_profile": PROFILE,
    **summary,
    "strict_context": {
        "reference_baseline": strict_queue["reference_baseline"],
        "required_best_return_pct": strict_queue["required_best_return_pct"],
        "reference_return_pct": strict_queue["reference_return_pct"],
        "max_walkforward_drawdown_pct": strict_queue["max_walkforward_drawdown_pct"],
        "final_retained_count": final_contract["final_retained_count"],
    },
    "gate_gaps": gate_gaps.to_dict("records"),
}
(CARD_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps(summary_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(json.dumps(summary_payload, indent=2, sort_keys=True))
scores.head(12)

{
  "candidate_profile": "mode_conditioned_rebound_reward_train_v2",
  "decision": "drawdown_drift_reward_recipe_ready_for_short_training_screen",
  "drawdown_failure_capture": 0.8732394366197183,
  "experiment": "mode_conditioned_drawdown_drift_reward_recipe_screen_v1",
  "gate_gaps": [
    {
      "avg_position": 0.2602524490812267,
      "buyhold_return_pct": 58.093570616483994,
      "failure_type": "joint_tradeoff",
      "gate_pass": false,
      "max_dd_gap": 3.086304019655522,
      "max_drawdown_pct": 23.086304019655522,
      "profile": "mode_conditioned_rebound_reward_train_v2",
      "relative_gap": 38.01024397057564,
      "relative_return_pct": -8.010243970575637,
      "return_pct": 50.08332664590836,
      "rows": 252,
      "window": "wf_2020"
    },
    {
      "avg_position": 0.0910703341999008,
      "buyhold_return_pct": -86.58807173239427,
      "failure_type": "pass",
      "gate_pass": true,
      "max_dd_gap": 0.0,
      "max_drawdown_pct": 15.9006974847681,
  

,recipe,recipe_ready,reward_recipe_loss,drawdown_failure_capture,trend_underparticipation_capture,overreach_share,active_rows,drawdown_breach_rows,controllable_drawdown_breach_rows,inactive_drawdown_breach_rows,...,trend_captured_rows,pressure_burden,drawdown_penalty,drawdown_threshold_pct,position_threshold,trend_floor_penalty,trend_floor_position,trend_lookback,min_buyhold_return_pct,reward_objective_patch_json
0,dd1.5_thr18_pos0.15_tf1_floor0.35_lb21,True,2.355216,0.873239,1.0,0.0,226,144,71,73,...,164,1.177608,1.5,18.0,0.15,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
1,dd1.5_thr18_pos0.15_tf1_floor0.25_lb21,True,2.476151,0.873239,1.0,0.0,153,144,71,73,...,91,1.238076,1.5,18.0,0.15,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
2,dd1.5_thr18_pos0.1_tf1_floor0.35_lb21,True,2.517677,0.873239,1.0,0.0,226,144,71,73,...,164,1.258838,1.5,18.0,0.10,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
3,dd2_thr18_pos0.15_tf1_floor0.35_lb21,True,2.607745,0.873239,1.0,0.0,226,144,71,73,...,164,1.303872,2.0,18.0,0.15,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."
4,dd1.5_thr18_pos0.05_tf1_floor0.35_lb21,True,2.680137,0.873239,1.0,0.0,226,144,71,73,...,164,1.340069,1.5,18.0,0.05,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
5,dd1.5_thr18_pos0.1_tf1_floor0.25_lb21,True,2.716125,0.873239,1.0,0.0,153,144,71,73,...,91,1.358063,1.5,18.0,0.10,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
6,dd2_thr18_pos0.1_tf1_floor0.35_lb21,True,2.824359,0.873239,1.0,0.0,226,144,71,73,...,164,1.412179,2.0,18.0,0.10,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."
7,dd2_thr18_pos0.15_tf1_floor0.25_lb21,True,2.849167,0.873239,1.0,0.0,153,144,71,73,...,91,1.424583,2.0,18.0,0.15,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."
8,dd1.5_thr15_pos0.15_tf1_floor0.35_lb21,True,2.926424,1.000000,1.0,0.0,235,144,71,73,...,164,1.463212,1.5,15.0,0.15,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
9,dd1.5_thr18_pos0.05_tf1_floor0.25_lb21,True,2.956099,0.873239,1.0,0.0,153,144,71,73,...,91,1.478050,1.5,18.0,0.05,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."


## Conclusion

- Ran: candidate-specific drawdown-drift reward recipe screen for `mode_conditioned_rebound_reward_train_v2` across `wf_2020`, `wf_2022`, and `wf_2023_2024`.
- Decision: `drawdown_drift_reward_recipe_ready_for_short_training_screen`; selected recipe `dd1.5_thr18_pos0.15_tf1_floor0.35_lb21` is ready for a short fold-local BC/PPO training screen.
- Key metrics: controllable drawdown capture `0.873`, raw drawdown capture `0.431`, trend underparticipation capture `1.000`, overreach share `0.000`.
- Risk note: `73` of `144` drawdown-breach rows were no-position drawdown tails, so they remain risk evidence but should not block reward-pressure readiness.
- Promotion decision: no promotion and no actor retention; final retained actor count remains `0`.
- Next action: run `short_foldlocal_bc_or_ppo_training_screen` with `reward_objective_patch.json`, then evaluate with the official walk-forward gate.
- Artifact links: `gate_gaps.csv`, `reward_recipe_scores.csv`, `reward_objective_patch.json`, `selection.json`, `summary.json`, and `reward_recipe_review.md`.
